# Using Libra built-in functions to compute overlaps in Kohn-Sham and excited states basis

In this super tutorial, we will learn how to use the Libra's built-in functions to compute the overlap integrals using different methods.

## Table of contents
<a name="toc"></a>
1. [Importing needed libraries](#import)
2. [Overview of required files](#required_files)
3. [Computing KS overlap integrals using Gaussian `cube` files](#cube)
4. [How to use Libint functions in Libra?](#libint_in_libra)\
  4.1 [Creating an integration shell](#create_shell)\
  4.2 [Computing the overlap integral using `compute_overlaps` function](#compute_overlaps)\
  4.3 [Computing moment integrals using `compute_emultipole3` function](#compute_emultipole3)\
  4.4 [Let's build a function](#build_a_function)\
  4.5 [Computing Kohn-Sham molecular orbital overlaps using molden files](#overlap_molden)\
5. [Excited states overlap integrals](#exc_ovlp)\
  5.1 [Slater determinant definition](#SD_def)\
  5.2 [What is step 3 workflow in Libra? The TiO2 unit cell example](#step3_workflow)
  
   

## Learning objectives

* To understand the underlying algorithm in step 2 and 3 calculations
* To be able to use the functions used in step 2 and 3 modules
* To be able to debug the Libra code for overlap calculations using the methods presented in this tutorial
* To be able to implement new algorithms and modifying the underlying code related to step 2 and 3


## 1. Importing needed libraries <a name="import"></a>
[Back to TOC](#toc)

Let's import the following modules.

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import time

In [3]:
from liblibra_core import *

In [4]:
from libra_py import units
from libra_py import data_conv
from libra_py import molden_methods
from libra_py import cube_file_methods
import libra_py.packages.cp2k.methods as CP2K_methods

In [5]:
from libra_py.workflows.nbra import mapping
from libra_py.workflows.nbra import mapping3
from libra_py.workflows.nbra import step2
from libra_py.workflows.nbra import step2_many_body
from libra_py.workflows.nbra import step3_many_body
from libra_py.workflows.nbra import step3

## 2. Overview of required files <a name="required_files"></a>
[Back to TOC](#toc)

The files that we will be using for this super tutorial are as follows:

* `HOMO.cube`, `LUMO.cube`, `TiO2_unit_cell.molden`, `St_ks_1200.npz`, and a set of log files, `all_logfiles/`, that contain the TD-DFT data from CP2K calculation for the unit cell of TiO2 (see [this tutorial](../7_step2_cp2k/1_DFT/2_hpc/1_example_TiO2)). 

You need to first untar the file `data.tar.bz2` to extract these files.

## 3. Computing KS overlap integrals using Gaussian `cube` files<a name="cube"></a>
[Back to TOC](#toc)


In this part we can see how we can use the `cube_file_methods` functions to read the data in cube files and integrate them. This approach was first used to interface CP2K, Gaussian 09, and DFTB+ with Libra. Currently, this approach is deprecated but it can be useful for small systems. These files contain the information for a cubic grid of the simulation cell. They can be large for large systems up to a couple of GBs.

Let's read a sample cube file for a TiO2 unit cell:

In [10]:
homo_cube = cube_file_methods.read_cube('HOMO.cube')
print(homo_cube)

[-3.1842e-16 -6.2411e-17  3.9750e-16 ...  6.4707e-13  1.0841e-12
  1.5415e-12]


This is how we can compute the volume element $dv$ which is used in integration:

In [12]:
volume_element = cube_file_methods.grid_volume('HOMO.cube')
print(volume_element)

0.0026481024175524755


And we can integrate between cubes using:

In [13]:
int_homo_homo = cube_file_methods.integrate_cube(homo_cube, homo_cube, volume_element)
print(int_homo_homo)

1.0000023905586597


You can see the normalization of the HOMO cube file. 

Excercise: Try this for `LUMO.cube` and compute the overlap between HOMO and LUMO levels.

## 4. How to use Libint functions in Libra? <a name="libint_in_libra"></a>
[Back to TOC](#toc)

There are lots of C++ implementation of the Libint code inside Libra. We have to first provide a (set) of integration shells.

### 4.1 Creating an integration shell<a name="create_shell"></a>

This shell includes the coordinate, spherical or Cartesian coordinate representation (via a boolean flag called `is_spherical`), angular momentum value, exponents and contraction coefficients of the Gaussian type orbital basis functions (see [this](https://www.cp2k.org/basis_sets)):

$$\varphi_i(\vec{r}) = R_i(r) \cdot Y_{l_i, m_i}(\theta, \phi)$$

$$R_i(r) = r^{l_i} \sum_{j=1}^{N} c_{ij} \cdot \exp(-\alpha_j \cdot r^2)$$

The functions used for this purpose are `initialize_shell` and `add_to_shell` which are part of `liblibra_core` module. The first one initializes an integration shell, and the other adds more shells to the initialized shell. This gives us the flexibility when reading turning the `molden` file into a set of integration shells. The coordinate are required to be defined as `VECTOR` type and the values of exponents and contraction coefficients are turned into C++ double type using `Py2Cpp_double` function. The exponents and the coefficients are passed as a `list` to both functions.

In [14]:
# Define the coordinate
coord = VECTOR(1.0, 0.000, 0.000)
# Spherical coordinate flag
is_spherical = True
# Angular momentum value
l_val = 0
# Exponent
exp = Py2Cpp_double([4.0])
# Coefficient
coeff = Py2Cpp_double([-0.5])
# Initialize a shell
shell = initialize_shell( int(l_val), is_spherical, exp, coeff, coord)
# Now add whatever remained from the basis set to this shell 
# as many times as needed
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)

You can check the information about the basis set via `nbasis(shell)` and `print_shell(shell)` function:

In [15]:
print('number of basis functions for this shell:',nbasis(shell))
# Let's print the shell itself
print_shell(shell)

number of basis functions for this shell: 4

	Shells are:
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4


### 4.2 Computing the overlap integral using `compute_overlaps` function <a name="compute_overlaps"></a>


Now let's compute the atomic orbital overlap matrix. This is done using the `compute_overlaps` function which computes the overlap between two different integration shells and return a `MATRIX`. The number of processors is defined in `nprocs`.

In [17]:
nprocs = 4 # Set the number of processors
A = compute_overlaps(shell, shell, nprocs) # Compute the overlap matrix
A.show_matrix()

1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



For practical purposes, turn the `MATRIX` into a `numpy` array using `data_conv.MATIX2nparray` function.

In [18]:
# Or turn it into a numpy array
data_conv.MATRIX2nparray(A).real

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

### 4.3 Computing moment integrals using `compute_emultipole3` function <a name="compute_emultipole3"></a>

The components that are retrieved via the `compute_emultipole3` function are as follows:

```python
    all_components = ['S', 'x', 'y', 'z', 'x2', 'xy', 'xz', 'y2', 'yz', 'z2',
                      'x3', 'x2y', 'x2z', 'xy2', 'xyz', 'xz2', 'y3', 'y2z', 'yz2', 'z3']
```

In [20]:
A = compute_emultipole3(shell, shell, 4)
# We have 20 components from overlap S, to dipole and quadrupole
print(len(A))

20


There are 20 components of integrals. This also includes overlap integral, `S`. They are stored in the order shown in `all_components` above.

In [22]:
# Let's print overlap, x, y, z, and x2
print('overlap:')
A[0].show_matrix()

overlap:
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



In [23]:
print('x:')
A[1].show_matrix()

x:
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



In [24]:
print('y:')
A[2].show_matrix()

y:
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   



In [25]:
print('z:')
A[3].show_matrix()

z:
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   



In [26]:
print('x2:')
A[4].show_matrix()

x2:
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   



### 4.4 Let's build a function <a name="build_a_function"></a>

We can try to make a function like as follows (this is not part of Libra but just a test function):

In [27]:
def create_shell_atom(coord, is_spherical, l_vals, exps, coeffs):
    """
    This function creates a set of main shells, each are separate, for all atoms in the system.
    """
    a = VECTOR(coord[0], coord[1], coord[2])
    for c1 in range(len(l_vals)):
        if c1==0:
            shell = initialize_shell(int(l_vals[c1]), is_spherical, Py2Cpp_double(exps[c1]), Py2Cpp_double(coeffs[c1]), a)
        else:
            add_to_shell(shell, int(l_vals[c1]), is_spherical, Py2Cpp_double(exps[c1]), Py2Cpp_double(coeffs[c1]), a)
            
    return shell

With this, we can create a set of integration shell related to an atom. This can be useful when you want to interface another software package, like **ORCA** or **OpenMolcas** or **QChem**, with Libra.

In [28]:
# Coordinate
coord = [1.0, 0.0, 0.0]
# Define the angular momentum values as appear in a basis set file
l_vals = [0,0,1,1,2]
# The list of exponents for each angular momentum value
exps = [[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],
       [12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],
       [12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761]]
# And their corresponding coefficients
coeffs = [[-0.060191,-0.129598,0.118176,0.462964,0.450354,0.092716,-0.000256],[0.065739,0.110886,-0.053732,-0.572671,0.186760,0.387201,0.003826],
         [0.036544,0.120928,0.251094,0.352640,0.294709,0.173040,0.009726],[-0.034211,-0.120620,-0.213719,-0.473675,0.484848,0.717466,0.032499],
         [0.014807,0.068186,0.290576,1.063344,0.307656,0.318347,-0.005772]]
# Create the integration shell
shell = create_shell_atom(coord, is_spherical, l_vals, exps, coeffs)
print('number of basis sets for this shell:', nbasis(shell))

number of basis sets for this shell: 13


Let's print the shell information

In [29]:
print_shell(shell)


	Shells are:
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=0,sph=1}
  12.015955 -0.28449995
  5.1081500 -0.32249868
  2.0483980 0.14819104
  0.83238200 0.29547527
  0.35231600 0.15082913
  0.14297700 0.015788331
  0.046761000 -1.8853185e-05


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=0,sph=1}
  12.015955 0.72562543
  5.1081500 0.64438457
  2.0483980 -0.15734917
  0.83238200 -0.85352822
  0.35231600 0.14606758
  0.14297700 0.15397733
  0.046761000 0.00065800393


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=1,sph=1}
  12.015955 1.1921517
  5.1081500 1.3541725
  2.0483980 0.89726731
  0.83238200 0.40883918
  0.35231600 0.11664777
  0.14297700 0.022184355
  0.046761000 0.00030839447


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=1,sph=1}
  12.015955 -1.0623948
  5.1081500 -1.2857933
  2.0483980 -0.72699821
  0.83238200 -0.52276451
  0.35231600 0.18268099
  0.14297700 0.087560117
  0.046761000 0.00098095033




In [30]:
# Now let's compute the atomic orbital overlap matrix
nprocs = 4 # Set the number of processors
A = compute_overlaps(shell, shell, nprocs) # Compute the overlap matrix
A.show_matrix()

1.0000000   -0.088210134  0.0000000   -8.2888693e-18  0.0000000   0.0000000   1.2754768e-17  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
-0.088210134  1.0000000   0.0000000   -1.6600822e-17  0.0000000   0.0000000   3.2941720e-18  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   -1.6738102e-18  0.0000000   0.0000000   0.0000000   
-8.2888693e-18  -1.6600822e-17  0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   9.6637478e-19  0.0000000   0.0000000   -1.6738102e-18  0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.0000000   -1.6738102e-18  
0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.0000000   -1.5077029e-18  0.0000000   0.0000000   0.0000000   
1.2754768e-17

And now let's see the other integral components:

In [31]:
A = compute_emultipole3(shell, shell, 4)
# We have 20 components from overlap S, to dipole and quadrupole
print(len(A))
# Let's print overlap, x, y, z, and x2
print('x:')
A[1].show_matrix()

20
x:
1.0000000   -0.088210134  0.0000000   0.67340060  0.0000000   0.0000000   0.31935015  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
-0.088210134  1.0000000   0.0000000   0.39866373  0.0000000   0.0000000   1.2570647   0.0000000   5.5511151e-17  0.0000000   0.0000000   5.5511151e-17  0.0000000   
0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.60376629  0.0000000   0.0000000   0.0000000   
0.67340060  0.39866373  0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   -0.34858463  0.0000000   0.0000000   0.60376629  0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.0000000   0.60376629  
0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.0000000   0.52270802  0.0000000   0.0000000   0.0000000   
0.31935015  1.2570647   0.0000000   0.176

### 4.5 Computing Kohn-Sham molecular orbital overlaps using molden files  <a name="overlap_molden"></a>

Here, we perform the overlap calculations using the molecular orbital coefficients. In CP2K, the basis set is of Gaussian type orbitals which are not orthonormal. We need to compute the atomic orbital overlap matrix between different types of GTOs and then use the molecular orbital coefficients to compute the molecular orbital overlaps:

$$\langle\phi_i|\phi_j\rangle = \sum_{a_{i}=0}^{N}\sum_{b_{j}=0}^{N}c_{a_i}^*c_{b_j}^*\langle\psi_{a_{i}}|\psi_{b_{j}}\rangle$$

### How does a `molden` file look like  <a name="molden_file"></a>

This is how a `molden` file format looks like:
```
 [Molden Format]
 [Atoms] AU
 Ti       1      22       4.396705       4.396705       2.805490
 Ti       2      22       0.000000       0.000000       0.000000
 O        3       8       1.718408       7.075002       2.805490
 O        4       8       7.075002       1.718408       2.805490
 O        5       8       2.678297       2.678297       0.000000
 O        6       8       6.115113       6.115113       0.000000
 [GTO]
        1       0
                         s       6    1.00
                                                          7.88456993        0.00475058
                                                          3.89469846        0.49950386
                                                          1.51358883       -0.66499588
                                                          0.59676808       -0.72604457
                                                          0.22222213       -0.02901108
                                                          0.07707846        0.07517175
                         s       6    1.00
                                                          7.88456993       -0.00269070
....
 [MO]
Ene=   -2.1223307907E+00
Spin= Alpha
Occup=   2.0000000
     1 -7.01185407E-01
     2  3.22437005E-02
     3  9.91737658E-03
....
```

We have a sample molden file for a **periodic** structure, the TiO2 unit cell from the previous steps. The reason we select this is to show how to use the traslational vectors in computation of the molecular overlap integrations.
The main function in Libra that turns a `molden` file into a Libint shell using the above functions is `molden_file_to_libint_shell` in the `libra_py.molden_methods` module. It's documentation is as follows:

```python
def molden_file_to_libint_shell(molden_filename: str, is_spherical: bool, is_periodic=False,
                                cell=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]), R_vec=np.array([0, 0, 0])):
    """
    This function gets the molden file and returns the shell for use with
    libint to compute the atomic orbital overlaps.

    Args:

        molden_filename (string): The name of the molden file.

        is_spherical (bool): The Gaussian cartesian or spherical orbital flag.
        
        is_periodic (bool): Whether the cell is periodic or not.
        
        cell (nparray): If the cell is periodic, what is the cell dimensions.
        
        R_vec (nparray): What are the translational vector

    Returns:

        shell (shell data type from libint): The shell containing the exponents and contraction
                                             coefficients for each atom, their coordinates in Bohr,
                                             and their angular momentum values.
        l_vals_all (list): The angular momentum values for all the atoms and basis set in order.
                           This is used later to resort the eigenvectors in molden file.

    """
```

Let's see how it operates. Before that, since this system is periodic, we need to define a set of translational vectors from CP2K `&CELL` input or from the cell information in the coordinate file (`POSCAR` or `CIF` etc):
```python
params['A_cell_vector'] = [4.6532721519, 0.0000000000, 0.0000000000]
params['B_cell_vector'] = [0.0000000000, 4.6532721519, 0.0000000000]
params['C_cell_vector'] = [0.0000000000, 0.0000000000, 2.9692029953]
params['periodicity_type'] = 'XYZ'
```
and the translational vectors are defined using `CP2K_methods.generate_translational_vectors` function:
```python
# Set the origin for generating the translational vectors (for creating Bloch type functions)
origin = [0,0,0]
tr_vecs = params['translational_vectors'] = CP2K_methods.generate_translational_vectors(origin, [2,2,2],
                                                                                        params['periodicity_type'])
```
the variable `[2,2,2]` defines how many periodic images to reproduce in each of the X, -X, Y, -Y, Z, and -Z directions respectively. It can be changed and you can see that we are producing $(2\times2+1)^3-1=125-1=124$ translational vectors (excluding $(0,0,0)$ which is the central cell itself.

Let's try this for `[1,1,1]` i.e. $(2\times1+1)^3-1=26$ translational vectors.

In [32]:
params_1 = {}
params_1['A_cell_vector'] = [4.6532721519, 0.0000000000, 0.0000000000]
params_1['B_cell_vector'] = [0.0000000000, 4.6532721519, 0.0000000000]
params_1['C_cell_vector'] = [0.0000000000, 0.0000000000, 2.9692029953]
params_1['periodicity_type'] = 'XYZ'
origin = [0,0,0]
tr_vecs = params_1['translational_vectors'] = CP2K_methods.generate_translational_vectors(origin, [1,1,1],
                                                                                        params_1['periodicity_type'])
print('The translational vectors for the current periodic system are:\n')
print(tr_vecs)
print(F'Will compute the S^AO between R(0,0,0) and {tr_vecs.shape[0]} translational vectors')

The translational vectors for the current periodic system are:

[[-1 -1 -1]
 [-1 -1  0]
 [-1 -1  1]
 [-1  0 -1]
 [-1  0  0]
 [-1  0  1]
 [-1  1 -1]
 [-1  1  0]
 [-1  1  1]
 [ 0 -1 -1]
 [ 0 -1  0]
 [ 0 -1  1]
 [ 0  0 -1]
 [ 0  0  1]
 [ 0  1 -1]
 [ 0  1  0]
 [ 0  1  1]
 [ 1 -1 -1]
 [ 1 -1  0]
 [ 1 -1  1]
 [ 1  0 -1]
 [ 1  0  0]
 [ 1  0  1]
 [ 1  1 -1]
 [ 1  1  0]
 [ 1  1  1]]
Will compute the S^AO between R(0,0,0) and 26 translational vectors


Before going into calculations, let's increase the number of translational vectors to have more accuracy:

In [33]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [2,2,2], params_1['periodicity_type'])

**Important note:** We need to turn the cell vectors to atomic units first using `units.Angst`:

In [34]:
cell = []
cell.append(params_1['A_cell_vector'])
cell.append(params_1['B_cell_vector'])
cell.append(params_1['C_cell_vector'])
cell = np.array(cell) * units.Angst

Then, we generate the integral shell and the angular momentum values related to each basis functions as they appear in the molden file. Let's consider the central cell for now i.e. `R_vec=np.array([0, 0, 0])`.

In [40]:
shell_1, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array([0, 0, 0]))
print(shell_1)
print('number of basis functions:',nbasis(shell_1))
print(l_vals)

number of basis functions: 104
[0, 0, 0, 1, 1, 2, 2, 3, 0, 0, 0, 1, 1, 2, 2, 3, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2]


 You can `print_shell(shell_1)` yourself (it would be a lengthy output).

Now, let's compute the overlap between shells using `compute_overlaps` function.

In [41]:
# number of processors to use
nprocs = 4
# Atomic orbital overlap matrix
AO = compute_overlaps(shell_1, shell_1, nprocs)
print(AO)

We need to turn this `MATRIX` object to `nparray` first using:

In [42]:
AO_numpy = data_conv.MATRIX2nparray(AO)
print(AO_numpy[0:10,0])

[ 1.00000000e+00+0.j  6.42868266e-02+0.j -2.08855695e-01+0.j
  5.20933915e-16+0.j -3.77792680e-17+0.j -3.77792680e-17+0.j
 -2.53972475e-16+0.j -1.08710819e-16+0.j -1.08710819e-16+0.j
  0.00000000e+00+0.j]


To generate the atomic orbital overlap matrix for all translational vectors we need a `for` loop as follows:

In [43]:
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)

#### Sorting the indices

Different software use different convention to present the angular momentum components. The version of Libint that is installed via `conda` is for `Psi4` package and not CP2K. So we need to resort the eigenvectors (or the atomic orbital matrix itself) accordingly. This is done using `CP2K_methods.resort_molog_eigenvectors` function. 

In [44]:
new_indices = CP2K_methods.resort_molog_eigenvectors(l_vals)
print(new_indices)

[0, 1, 2, 4, 5, 3, 7, 8, 6, 11, 12, 10, 13, 9, 16, 17, 15, 18, 14, 22, 23, 21, 24, 20, 25, 19, 26, 27, 28, 30, 31, 29, 33, 34, 32, 37, 38, 36, 39, 35, 42, 43, 41, 44, 40, 48, 49, 47, 50, 46, 51, 45, 52, 53, 55, 56, 54, 58, 59, 57, 62, 63, 61, 64, 60, 65, 66, 68, 69, 67, 71, 72, 70, 75, 76, 74, 77, 73, 78, 79, 81, 82, 80, 84, 85, 83, 88, 89, 87, 90, 86, 91, 92, 94, 95, 93, 97, 98, 96, 101, 102, 100, 103, 99]


Now, we read the eigenvectors. Due to a specific writing format of the eigenvectors in molden files, we need to know what is the number of basis functions that appear in an integration shells i.e. all number of atomic orbitals, which is done using `nbasis` function from `liblibra_core` module.

In [47]:
number_of_basis_functions = nbasis(shell_1)
print(number_of_basis_functions)
eigenvectors, energies = molden_methods.eigenvectors_molden('TiO2_unit_cell.molden', number_of_basis_functions, l_vals)
print(len(eigenvectors))
print(eigenvectors[0].shape)

104
74
(104,)


For unrestricted spin calculations, the eigenvectors are sorted in a different way. Please be careful of parsing the `molden` file. It might be the case that the alpha and beta orbitals are written either in a sequential format or they are mixed. The way CP2K outputs these eigenvectors is that the first half of the eigenvectors are the alpha MOs and the next half are beta MOs.

Now, let's sort the eigenvectors:

In [48]:
eigenvectors_1 = []
for j in range(len(eigenvectors)):
    # the new and sorted eigenvector
    eigenvector_1 = eigenvectors[j]
    eigenvector_1 = eigenvector_1[new_indices]
    # append it to the eigenvectors list
    eigenvectors_1.append(eigenvector_1)
eigenvectors_1 = np.array(eigenvectors_1)

And finally, computing the MO overlap matrix:

In [49]:
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])
print(np.diag(MO_overlap))

[0.99999961+0.j 1.00000041+0.j 0.99999999+0.j 1.        +0.j
 1.00000031+0.j 1.00000029+0.j 0.99998989+0.j 1.00001294+0.j
 0.99995745+0.j 1.00003095+0.j 1.00000712+0.j 1.00000716+0.j
 0.99999744+0.j 0.99999625+0.j 0.99999708+0.j 1.00001662+0.j
 1.00001658+0.j 0.9999872 +0.j 0.99998869+0.j 0.99908439+0.j
 1.00007521+0.j 0.9999783 +0.j 0.99997998+0.j 1.00001046+0.j
 0.9999867 +0.j 1.00001724+0.j 0.99999472+0.j 1.00006086+0.j
 1.00006085+0.j 1.000052  +0.j 0.99999583+0.j 0.99958757+0.j
 0.99958764+0.j 0.9995215 +0.j 1.00021321+0.j 1.00004889+0.j
 1.00005065+0.j 0.99959955+0.j 1.00496174+0.j 0.88632026+0.j
 1.01034228+0.j 1.00116205+0.j 1.0011538 +0.j 1.09927545+0.j
 0.98099997+0.j 1.01329743+0.j 1.05242962+0.j 1.05242983+0.j
 1.00020224+0.j 1.00008737+0.j 0.97469219+0.j 1.04710326+0.j
 0.99587122+0.j 0.95245193+0.j 1.02667602+0.j 1.02667578+0.j
 1.00006537+0.j 1.00498198+0.j 0.99969375+0.j 0.9996906 +0.j
 1.01092995+0.j 0.99595327+0.j 0.99595416+0.j 0.98170708+0.j
 1.00778589+0.j 1.007790

Let's compute the determinant of this matrix to check its orthonormality:

In [50]:
print(np.linalg.det(MO_overlap))

(0.9068820174820696+0j)


It's still not orthonormal but close to it. To further increase the accuracy, you need to include more number of translational vectors. Let's continue with `[3,3,3]` for translational vectors:

In [51]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [3,3,3], params_1['periodicity_type'])
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])

In [52]:
print(np.linalg.det(MO_overlap))

(1.000740369408316+0j)


In [53]:
print(np.diag(MO_overlap))

[1.        +0.j 1.00000001+0.j 1.        +0.j 1.00000001+0.j
 1.00000001+0.j 1.00000001+0.j 1.00000002+0.j 1.00000003+0.j
 1.        +0.j 1.00000023+0.j 0.99999996+0.j 0.99999998+0.j
 1.0000001 +0.j 1.00000006+0.j 1.00000006+0.j 1.0000001 +0.j
 1.00000005+0.j 0.99999993+0.j 0.99999999+0.j 0.99999434+0.j
 1.00000036+0.j 0.99999986+0.j 0.99999986+0.j 1.00000003+0.j
 0.99999988+0.j 0.99999993+0.j 1.        +0.j 1.00000006+0.j
 1.00000005+0.j 1.00000055+0.j 0.99999992+0.j 0.99999897+0.j
 0.99999906+0.j 0.99999887+0.j 0.99999947+0.j 1.00000014+0.j
 1.00000027+0.j 0.99999987+0.j 1.00004873+0.j 0.99966431+0.j
 1.00001186+0.j 1.00000076+0.j 1.00000059+0.j 1.0002019 +0.j
 0.99994581+0.j 1.00000945+0.j 1.00006596+0.j 1.000066  +0.j
 0.99999981+0.j 0.9999997 +0.j 0.99992055+0.j 1.00047491+0.j
 0.99999309+0.j 1.00009738+0.j 1.00002634+0.j 1.00002643+0.j
 0.99999983+0.j 1.00003122+0.j 0.99999924+0.j 0.99999906+0.j
 1.0000136 +0.j 0.99998816+0.j 0.99998824+0.j 0.99994193+0.j
 1.0000264 +0.j 1.000026

You can see that the determinant is `1.00074` (accuracy up to $10^{-3}$). Let's try one more by increasing `[4,4,4]`:

In [54]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [4,4,4], params_1['periodicity_type'])
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])
print(np.linalg.det(MO_overlap))

(0.9999897428194451+0j)


which has a higher accuracy of $10^{-4}$.

You can do the same thing to compute the overlap between different geoemtries using the same procedure. This procedure is implemented in step 2 of Libra for computation of the time-overlap of molecular orbitals of two consecutive geometries. See [this tutorial](../7_step2_cp2k).

#### Computing the integrals using `compute_emultipole3` for periodic system

Let's try the `compute_emultipole3` function and compute the dipole moment operator matrix in the molecular orbital basis: 

In [55]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [4,4,4], params_1['periodicity_type'])
A = compute_emultipole3(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    A_periodic = compute_emultipole3(shell_1, shell_periodic, nprocs)
    # Since there are 20 components
    for i in range(20):
        A[i] += A_periodic[i]

You can simply use a list like `['S','x','y','z','x2']` and turn it into a set of indices and labels using `step2.component_to_index` function as follows:

In [56]:
emultipole_index, emultipole_labels = step2.component_to_index(['S','x','y','z','x2'])
print(emultipole_index, emultipole_labels)

[0, 1, 2, 3, 4] ['S', 'x', 'y', 'z', 'x2']


In [57]:
# Now turn each component into numpy array
MO_matrices = []
for i in emultipole_index:
    A_numpy = data_conv.MATRIX2nparray(A[i])
    MO_matrix = np.linalg.multi_dot([eigenvectors_1, A_numpy, eigenvectors_1.T])
    print('The first 10 diagonal element of the', emultipole_labels[i], 'component:')
    print(np.diag(MO_matrix)[0:10])
    MO_matrices.append(MO_matrix)

The first 10 diagonal element of the S component:
[0.99999992+0.j 0.99999981+0.j 0.99999981+0.j 0.99999897+0.j
 1.00000207+0.j 1.00000074+0.j 1.00000286+0.j 1.00000285+0.j
 1.00000162+0.j 1.00000683+0.j]
The first 10 diagonal element of the x component:
[2.21729555+0.j 2.22771052+0.j 4.14499858+0.j 0.38157711+0.j
 0.34703172+0.j 4.1332175 +0.j 2.24936438+0.j 2.22455187+0.j
 4.26708444+0.j 4.28353403+0.j]
The first 10 diagonal element of the y component:
[2.21699672+0.j 2.22730597+0.j 4.14437707+0.j 0.38141168+0.j
 0.3501262 +0.j 4.1332436 +0.j 2.24906972+0.j 2.22441138+0.j
 4.23528522+0.j 4.24728355+0.j]
The first 10 diagonal element of the z component:
[1.39980762+0.j 1.39757285+0.j 2.63123383+0.j 0.1680248 +0.j
 0.21709888+0.j 2.57120654+0.j 1.40430813+0.j 1.4012134 +0.j
 1.33295171+0.j 1.44089215+0.j]
The first 10 diagonal element of the x2 component:
[10.29190049+0.j 10.37187141+0.j 18.89080314+0.j  2.53301939+0.j
  2.33425247+0.j 18.81979315+0.j 10.31251228+0.j 10.15721735+0.j
 24

Let's check the determinant of the MO overlap matrix using emultipole3 approach:

In [58]:
print('Determinant of the overlap matrix from compute_emultipole3 function:', np.linalg.det(MO_matrices[0]))

Determinant of the overlap matrix from compute_emultipole3 function: (0.9801204716209111+0j)


## 5. Excited states overlap integrals<a name="exc_ovlp"></a>

[Back to TOC](#toc)

### 5.1 Slater determinant definition <a name="SD_def"></a>

The following are adopted from "Modern Quantum Chemistry" book by Szabo and Ostlund.

The spin-orbital is defined as follows:

$$
\chi(\mathbf{x}) = 
\begin{cases}
\psi_{\alpha}(\mathbf{r}) \alpha(\omega) \\
\text{or} \\
\psi_{\beta}(\mathbf{r}) \beta(\omega)
\end{cases}
$$

which is a product of the spatial orbital and the $\alpha(\omega)$ or $\beta(\omega)$ spin function. 

Using this set of spin-orbitals, one can define a Slater-determinant (SD) as:

$$
\Psi(\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_N) = (N!)^{-1/2}
\begin{vmatrix}
\chi_i(\mathbf{x}_1) & \chi_j(\mathbf{x}_1) & \cdots & \chi_k(\mathbf{x}_1) \\
\chi_i(\mathbf{x}_2) & \chi_j(\mathbf{x}_2) & \cdots & \chi_k(\mathbf{x}_2) \\
\vdots & \vdots & \ddots & \vdots \\
\chi_i(\mathbf{x}_N) & \chi_j(\mathbf{x}_N) & \cdots & \chi_k(\mathbf{x}_N)
\end{vmatrix}
$$

So, **the molecular SD is formed from the spin-orbitals**. We generate the Kohn-Sham molecular orbital overlap and save them in two block formats, one for $\alpha$ and the other for $\beta$:

$$\mathbf{S_{KS}}=
\begin{bmatrix}
\boldsymbol{\alpha} & \mathbf{0} \\
\mathbf{0} & \boldsymbol{\beta}
\end{bmatrix}
$$



Let $(\Phi_A)$ and $(\Phi_B)$ be two normalized $(n)$-electron Slater determinants:
$$
\Phi_A(x_1, x_2, \ldots, x_n) = \frac{1}{\sqrt{n!}} \det[\phi_i^A(x_j)]
\\
\Phi_B(x_1, x_2, \ldots, x_n) = \frac{1}{\sqrt{n!}} \det[\phi_i^B(x_j)]
\\
$$
We want to compute the overlap:
$$
\langle \Phi_A | \Phi_B \rangle = \int dx_1 \cdots dx_n \, \Phi_A^*(x_1, \ldots, x_n) \Phi_B(x_1, \ldots, x_n)
$$

Write Each Determinant Using Permutations

Using the Leibniz formula for determinants:
$$
\Phi_A(x_1, \ldots, x_n) = \frac{1}{\sqrt{n!}} \sum_{P \in S_n} \text{sgn}(P) \prod_{i=1}^{n} \phi^A_{P(i)}(x_i)
\\
\Phi_B(x_1, \ldots, x_n) = \frac{1}{\sqrt{n!}} \sum_{Q \in S_n} \text{sgn}(Q) \prod_{i=1}^{n} \phi^B_{Q(i)}(x_i)
$$

The overlap of two SD is:
$$
\langle \Phi_A | \Phi_B \rangle = \frac{1}{n!} \sum_{P, Q} \text{sgn}(P) \text{sgn}(Q)
\int dx_1 \cdots dx_n \prod_{i=1}^n \phi_{P(i)}^{A*}(x_i) \phi_{Q(i)}^B(x_i)
$$

Since each \(x_i\) appears only in one term in the product, the total integral is a product of single-particle overlaps:
$$
\int dx_1 \cdots dx_n \prod_{i=1}^n \phi_{P(i)}^{A*}(x_i) \phi_{Q(i)}^B(x_i)
= \prod_{i=1}^n \langle \phi_{P(i)}^A | \phi_{Q(i)}^B \rangle
$$

It turns out that this sum over permutations with alternating sign and a product of overlaps is exactly the definition of a determinant:

$$
\langle \Phi_A | \Phi_B \rangle = \det(S)
$$

where

$$
S_{ij} = \langle \phi_i^A | \phi_j^B \rangle
$$


**So the $\det[]$ function has the $\text{sgn}()$ function in itself.** 

It was shown in previous tutorials that how the single particle SDs are defined (see for example [this tutorial](../8_step3)). Let's compute the overlap of the following `SD1` with `SD2` and `SD3` using `ovlp_arb` function procedure (examples from Alexey):

```python
SD1 = [1,-1,2,-2]
SD2 = [-1,1,2,-2]
SD3 = [1,2,-1,-2]
```

In [59]:
SD1 = [1,-1,2,-2]
SD2 = [-1,1,2,-2]
SD3 = [1,2,-1,-2]

The overlap matrix is a simple identity matrix:

$$
S_{KS}=\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

The function `mapping.sd2indx` turns the SD values into the KS matrix indices. 

In [60]:
S_ks = np.diag([1.0]*4)
print('S_ks:\n', S_ks)
sd1 = mapping.sd2indx(SD1, S_ks.shape[0], False, 0) # No use_minimal and no user_notation
sd2 = mapping.sd2indx(SD2, S_ks.shape[0], False, 0)
sd3 = mapping.sd2indx(SD3, S_ks.shape[0], False, 0)
print('sd1 indices (similar alpha and beta channel):', sd1)
print('sd2 indices (similar alpha and beta channel):', sd2)
print('sd3 indices (similar alpha and beta channel):', sd3)

S_ks:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
sd1 indices (similar alpha and beta channel): [0, 0, 1, 1]
sd2 indices (similar alpha and beta channel): [0, 0, 1, 1]
sd3 indices (similar alpha and beta channel): [0, 1, 0, 1]


But the beta spin channel indices are similar to alpha spin channel. This can be simply fixed via addition of the half of the KS matrix size, `int(S_ks.shape[0]/2)`, for the elements of the SD list where we have negative indices (beta channel).

In [61]:
# What about beta spin channels? We just need to shift the ones
# related to negative values in SD1 by data_dim/2
beta_indices_1 = np.where(np.array(SD1) < 0)
sd1 = np.array(sd1)
sd1[beta_indices_1] += int(S_ks.shape[0]/2)

beta_indices_2 = np.where(np.array(SD2) < 0)
sd2 = np.array(sd2)
sd2[beta_indices_2] += int(S_ks.shape[0]/2)

beta_indices_3 = np.where(np.array(SD3) < 0)
sd3 = np.array(sd3)
sd3[beta_indices_3] += int(S_ks.shape[0]/2)
print('sd1 with beta channel included:', sd1)
print('sd2 with beta channel included:', sd2)
print('sd3 with beta channel included:', sd3)

sd1 with beta channel included: [0 2 1 3]
sd2 with beta channel included: [2 0 1 3]
sd3 with beta channel included: [0 1 2 3]


Now, we can write the SD matrices and compute their determinant as follows:

In [64]:
Slater_determinant_matrix = S_ks[sd1,:][:,sd1]
print(Slater_determinant_matrix)
print(SD1, SD1)
print('det(Slater_determinant_matrix):', np.linalg.det(Slater_determinant_matrix))

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
[1, -1, 2, -2] [1, -1, 2, -2]
det(Slater_determinant_matrix): 1.0


In [65]:
Slater_determinant_matrix = S_ks[sd1,:][:,sd2]
print(Slater_determinant_matrix)
print(SD1, SD2)
print('det(Slater_determinant_matrix):', np.linalg.det(Slater_determinant_matrix))

[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
[1, -1, 2, -2] [-1, 1, 2, -2]
det(Slater_determinant_matrix): -1.0


In [66]:
Slater_determinant_matrix = S_ks[sd1,:][:,sd3]
print(Slater_determinant_matrix)
print(SD1, SD3)
print('det(Slater_determinant_matrix):', np.linalg.det(Slater_determinant_matrix))

[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]
[1, -1, 2, -2] [1, 2, -1, -2]
det(Slater_determinant_matrix): -1.0


In [67]:
Slater_determinant_matrix = S_ks[sd2,:][:,sd3]
print(Slater_determinant_matrix)
print(SD2, SD3)
print('det(Slater_determinant_matrix):', np.linalg.det(Slater_determinant_matrix))

[[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]
[-1, 1, 2, -2] [1, 2, -1, -2]
det(Slater_determinant_matrix): 1.0


### 5.2 What is step 3 workflow in Libra? The TiO2 unit cell example <a name="step3_workflow"></a>


We will regenerate the step 3 data for a TiO2 system. We will show all the steps that is done in the `libra_py/workflow/nbra/step3.py` file to generate the data required for computation of the time-overlap in the SD and MB basis. Different functions are applied and tested and the important variables are explained in details.

Let's start with a dictionary of parameters. The only parameters that are taken in the step 3 are as follows (see the other tutorial on step 3 for TiO2 system in  [step 3 folder](../8_step3) for more details of the parameters):

```python
params_mb_sd = {
          'lowest_orbital': 24-10, 'highest_orbital': 24+11, 'num_occ_states': 10, 'num_unocc_states': 10,
          'isUKS': 0, 'number_of_states': 10, 'tolerance': 0.0, 'verbosity': 0, 'use_multiprocessing': True, 'nprocs': 12,
          'is_many_body': True, 'time_step': 1.0, 'es_software': 'cp2k',
          'path_to_npz_files': os.getcwd()+'/../../7_step2_cp2k/1_DFT/2_hpc/1_example_TiO2/res',
          'logfile_directory': os.getcwd()+'/../../7_step2_cp2k/1_DFT/2_hpc/1_example_TiO2/all_logfiles',
          'path_to_save_sd_Hvibs': os.getcwd()+'/res-mb-sd-DFT',
          'outdir': os.getcwd()+'/res-mb-sd-DFT', 'start_time': 1200, 'finish_time': 1401, 'sorting_type': 'energy',
         }

step3.run_step3_sd_nacs_libint(params_mb_sd)
```

You can see that the function used in here is called `step3.run_step3_sd_nacs_libint` which takes the `params_mb_sd` parameter. 

Let's go in more details and step by step calculations where we show that these parameters are enough to generate the time-overlaps in the SD and MB basis. But before that, let's talk about selection of the active space and how it will affect the results of the calculations.

### Active spaces in step 3

For large systems, we need to define an active space for **KS molecular orbital basis**. Why? Because saving of the KS basis data using full active space is not efficient. In fact, due to separate procedure taken by Libra for the NBRA workflow, we need to store the data from step 2 and have them available (the better option is to perform these calculations on-the-fly but we currently don't have that workflow available in Libra). 

So, we need to save the file efficiently and hence select an active space that starts from the `lowest_orbital` to `highest_orbital` which both of these numbers start from 1. 

Before going into details of other active states and how we select them, let's define the parameters that are brough in `params_mb_sd`. 

`lowest_orbital` and `highest_orbital`: These two parameters are defined for step 2 calculations where we generate the data in the KS basis which are overlap, time-overlap and energy matrices. 

For the next parameters, let's talk about how we generate the single-particle excitation states basis. Well, in step 3 calculations, we have two different types of excited states basis. One is single-particle, the other is many-body which are obtained from TD-DFT calculations. 

If parameter `is_many_body` is set to `True` or `1`, the single-particle excitations are obtained from reading the `.log` files (in `logfile_directory`) which contain the TD-DFT data for all geometries from `start_time` to `finish_time`. The number of TD-DFT states that are read when this flag is on is defined by `number_of_states`. For each excited state, only the configurations that have higher amplitudes than `tolerance` are selected. Let's see a sample of the TD-DFT data from `'es_software': 'cp2k'` below:

```
 -------------------------------------------------------------------------------
 -                            Excitation analysis                              -
 -------------------------------------------------------------------------------
        State             Occupied              Virtual             Excitation
        number             orbital              orbital             amplitude
 -------------------------------------------------------------------------------
             1   0.60648 eV
                                40                   41              -0.973537
                                40                   44               0.156360
                                40                   42              -0.083794
                                40                   43               0.057608
                                39                   47              -0.051266
                                38                   47              -0.042105
                                40                   45              -0.036066
                                29                   41               0.034701
...........
```

On the other hand, sometimes, it is hard to perform TD-DFT calculations, so one needs to define the single-particle excited states themselves. But how to do this? The approach we adopt here is to generate this basis by exciting from the occupied orbitals (`num_occ_states` from the band-edge) to unoccupied orbitals (`num_unocc_states` from the band-edge). 

For example, if we have `'num_occ_states': 5` and `'num_unocc_states': 4`, all the single-particle excited state basis would be `20` which includes all excitation from occupied orbitals from the band edge (`HOMO to HOMO-4`) to unoccupied orbitals of `LUMO to LUMO+3`. In addition, we will consider the ground-state which is $$\Psi_{GS} = | \phi_{HOMO-4}^{\alpha} \phi_{HOMO-4}^{\beta} \phi_{HOMO-3}^{\alpha} \phi_{HOMO-3}^{\beta} \phi_{HOMO-2}^{\alpha} \phi_{HOMO-2}^{\beta} \phi_{HOMO-1}^{\alpha} \phi_{HOMO-1}^{\beta}\phi_{HOMO}^{\alpha} \phi_{HOMO}^{\beta} \rangle$$. 

This will be noted using `SD_GS=[1,-1,2,-2,3,-3,4,-4,5,-5]` and other excited state, for example the $$\phi_{HOMO}^{\alpha}\rightarrow\phi_{LUMO+1}^{\alpha}$$ is:

$$\Psi_{GS} = | \phi_{HOMO-4}^{\alpha} \phi_{HOMO-4}^{\beta} \phi_{HOMO-3}^{\alpha} \phi_{HOMO-3}^{\beta} \phi_{HOMO-2}^{\alpha} \phi_{HOMO-2}^{\beta} \phi_{HOMO-1}^{\alpha} \phi_{HOMO-1}^{\beta}\phi_{LUMO+1}^{\alpha} \phi_{HOMO}^{\beta} \rangle$$

which is generated in Libra as `SD=[1,-1,2,-2,3,-3,4,-4,7,-5]`. 

### Extracting TD-DFT data from CP2K log files

Let's see how to extract information of the excited state basis from log files of a CP2K calculations. We first need to define another dictionary for the `CP2K_methods.read_cp2k_tddfpt_log_file` function.

In [68]:
lowest_orbital = 40-19
highest_orbital = 40+20
params_tddft = {'number_of_states': 10, 'tolerance': 0.05, 
                'logfile_name': 'all_logfiles/step_1200.log', 'isUKS': False, 
                'lowest_orbital': lowest_orbital, 'highest_orbital': highest_orbital}
data = CP2K_methods.read_cp2k_tddfpt_log_file(params_tddft)

One can read the KS HOMO index from the CP2K output using `CP2K_methods.read_homo_index` function. It will retrieve two numbers for alpha and beta HOMO index if there are two different HOMO index in the CP2K logfiles.

In [70]:
ks_homo_index = CP2K_methods.read_homo_index(params_tddft['logfile_name'])
print('The HOMO index is:',ks_homo_index)

The HOMO index is: 40


The first argument of the `data` that was just extracted contains the excited states energies in eV.

In [71]:
data[0]

[0.60648,
 0.97928,
 1.27659,
 1.8829,
 2.14219,
 2.29289,
 2.33594,
 2.49111,
 2.59778,
 2.62325]

The second output contains the excited states configurations. Its length is `number_of_states` and each element of that contain a list which is formed from one or multiple lists, each representing a single-particle excitation. For example:

In [72]:
print('The length of the `data` variable:',len(data[1]))
print('The data variable:',data[1])
print('The number of single-particle excitations in the 3rd excited states:',len(data[1][2]))
print('The compositions of the single-particle excitations in the 3rd excited states:',data[1][2])

The length of the `data` variable: 10
The data variable: [[[40, 41]], [[40, 42]], [[40, 43], [40, 44]], [[40, 44], [40, 46]], [[40, 45], [40, 46]], [[39, 41], [39, 42], [40, 46], [37, 41]], [[40, 46], [40, 45], [39, 41]], [[38, 41], [39, 42], [40, 47], [37, 41]], [[40, 47], [37, 41], [38, 42]], [[39, 42], [38, 41], [39, 43], [38, 42], [37, 41]]]
The number of single-particle excitations in the 3rd excited states: 2
The compositions of the single-particle excitations in the 3rd excited states: [[40, 43], [40, 44]]


Whic means that the 3rd excited state has two configurations (each with amplitude more than `'tolerance': 0.02`), one is `40->43` (HOMO to LUMO+2) and the other `40->44` (HOMO to LUMO+3).

The third element of the `data` variable contains the configuration interaction coefficients related to each single-particle excitation in each excited states.

In [73]:
data[2]

[[-0.973537],
 [-0.971833],
 [0.938266, -0.247684],
 [-0.871286, 0.337412],
 [-0.868286, 0.355366],
 [0.798763, 0.313484, -0.28438, 0.278284],
 [-0.802833, -0.356074, -0.229352],
 [0.711404, -0.470105, 0.313769, 0.250202],
 [0.730334, -0.451852, 0.297089],
 [0.64084, 0.427694, -0.327131, -0.280067, -0.258855]]

And the last element, contains the spin information.

In [74]:
data[3]

[['alp'],
 ['alp'],
 ['alp', 'alp'],
 ['alp', 'alp'],
 ['alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp'],
 ['alp', 'alp', 'alp', 'alp', 'alp']]

Now, that we have defined our parameters, let's continue talking about active spaces. When running the calculations in step 3, we have a block matrix out of the full matrix of the KS molecular orbitals. This matrix, as is explained in [this tutorial](../7_step2_cp2k), is formed of four blocks: alpha and beta molecular orbitals with two zero blocks. 

**We only know the data dimension of this matrix** defined by `data_dim` in `step3.py` file. We need to find what is the index of the HOMO level index in this matrix reduced matrix.

If we have `'is_many_body': True`, then, after finding all the SDs from the TD-DFT data, the KS orbital indices will start from `min_band` to `max_band`. The value of these two variables is defined as follows:

```python
# Generate the excitation data
res = step3_many_body.get_step2_mb_sp_properties(params)

# The unique SDs in the TD-DFT results in all logfiles
sd_unique_basis = res[0]

# The target KS states in TD-DFT
sd_fstates = []

# The initial KS states in TD-DFT
sd_tstates = []
for i in range(len(sd_unique_basis)):
    sd_fstates.append(sd_unique_basis[i][0][0])
    sd_tstates.append(sd_unique_basis[i][0][1])

# The min_band present in the excitation
min_band = min(sd_fstates)

# The max_band present in the excitation
max_band = max(sd_tstates)

# number of occupied and unoccupied KS states
num_occ = ks_homo_index - min_band + 1
num_unocc = max_band - ks_homo_index  # + 1
```

otherwise the `min_band = 1` and `max_band = int(data_dim/2)`.

In [76]:
params_2 = {"logfile_directory": "all_logfiles",
            "es_software": "cp2k", "isUKS": 0, "number_of_states": 10,
            "tolerance": 0.02, "isnap": 1200, "fsnap": 1210, 
            'lowest_orbital': lowest_orbital, 'highest_orbital': highest_orbital}

res = step3_many_body.get_step2_mb_sp_properties(params_2)

This variable contains all possible excitations found in the log files from `'isnap': 1200` to `'fsnap': 1210`. The reason it is called "unique" is because all elements in this list are unique. As you can see above, this data is generated from `step3_many_body.get_step2_mb_sp_properties` function.

In [77]:
print(res[0])

[[[40, 41], 'alp'], [[40, 44], 'alp'], [[40, 42], 'alp'], [[40, 43], 'alp'], [[40, 45], 'alp'], [[40, 46], 'alp'], [[39, 41], 'alp'], [[39, 42], 'alp'], [[37, 41], 'alp'], [[40, 47], 'alp'], [[38, 41], 'alp'], [[35, 41], 'alp'], [[38, 42], 'alp'], [[39, 43], 'alp'], [[40, 49], 'alp'], [[37, 42], 'alp'], [[38, 43], 'alp'], [[36, 42], 'alp']]


Let's use the above function that is used in `step3.py`:

In [78]:
# The unique SDs in the TD-DFT results in all logfiles
sd_unique_basis = res[0]

# The target KS states in TD-DFT
sd_fstates = []

# The initial KS states in TD-DFT
sd_tstates = []
for i in range(len(sd_unique_basis)):
    sd_fstates.append(sd_unique_basis[i][0][0])
    sd_tstates.append(sd_unique_basis[i][0][1])

# The min_band present in the excitation
min_band = min(sd_fstates)

# The max_band present in the excitation
max_band = max(sd_tstates)

# number of occupied and unoccupied KS states
num_occ = ks_homo_index - min_band + 1
num_unocc = max_band - ks_homo_index  # + 1

print('min_band:', min_band)
print('max_band:', max_band)
print('num_occ:', num_occ)
print('num_unocc:', num_unocc)
ks_orbital_indicies = range(min_band, max_band + 1)
print('The range of the KS orbital indices (starting from 1):', list(ks_orbital_indicies))

min_band: 35
max_band: 49
num_occ: 6
num_unocc: 9
The range of the KS orbital indices (starting from 1): [35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


You can see above that we find how many occupied and unoccupied orbital are involved in the excitation analysis of these steps. Now is the time to create a variable, `ks_active_space` with which we are able to extract data (overlaps, time-overlaps, or energies) from the raw KS data available in step 2 files. This variable will contain the indices for both alpha and beta channels and is generated using a function called `make_active_space`.

The documentation of this function is as follows:

In [79]:
help(step3.make_active_space)

Help on function make_active_space in module libra_py.workflows.nbra.step3:

make_active_space(num_occ, num_unocc, data_dim, ks_homo_index, isUKS=0, num_occ_alpha=0, num_unocc_alpha=0, num_occ_beta=0, num_unocc_beta=0)
    This function makes an active space based on the number of occupied and
    unoccupied orbitals and the initial KS HOMO index. **Note that the ks_homo_index
    starts from 1.**
    
    Args:
    
        num_occ (integer): Number of occupied orbitals from HOMO
    
        num_unocc (integer): Number of unoccupied orbitals from LUMO
    
        data_dim (integer): The data dimension of the 'raw' overlap matrices
    
        ks_homo_index (list/int): The KS HOMO indexes (which starts from 1) of the 'raw' matrices.
        If it's a list, it should be in the format [homo_alpha, homo_beta]
    
        isUKS : request unrestricted spin calculation. If specified, instead of new_ks_homo_index
        returns (new_ks_homo_alpha_index, new_ks_homo_beta_index). Default i

Let's build one active space and see how it looks like. To do this, we need to find where was the index of the HOMO level in the raw files. We already extracted the HOMO level number from the CP2K outputs. We know that the `lowest_orbital = 40-19` and the `highest_orbital = 40+20` fom step 2 calculations. So, it would easily be defined as:

`npz_file_ks_homo_index` = `ks_homo_index` - `lowest_orbital` + 1

This variable is called `npz_file_ks_homo_index` and it starts from `1` (this is why we have `+1` at the end).

In [80]:
# What is the KS HOMO index in the raw npz files
npz_file_ks_homo_index = ks_homo_index - lowest_orbital + 1
# simply read a sample matrix and then see what is its dimension
sample_matrix = sp.load_npz('St_ks_1200.npz')
data_dim = sample_matrix.shape[0]
print('data_dim:', data_dim)
ks_active_space, ks_homo_index_1 = step3.make_active_space(num_occ, num_unocc, data_dim, npz_file_ks_homo_index)
print('npz_file_ks_homo_index:', npz_file_ks_homo_index)
print('ks_active_space:', ks_active_space)
print('ks_homo_index_1:', ks_homo_index_1)

data_dim: 80
npz_file_ks_homo_index: 20
ks_active_space: [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68]
ks_homo_index_1: 6


You can see that the new variable of `ks_homo_index_1` (which was `40` from CP2K log file) is now `6`.

Why? because we have 6 occupied orbitals. 

What about the indices? We can see that the `data_dim` is 80. Why? because the number of molecular orbitals considered from `lowest_orbital` to `highest_orbital` is `40` and for both alpha and beta spin channels it is `40*2=80`. Let's check the indices length and we can see that it is `15 = 6 (occupied orbitals) + 9 (unoccupied orbitals)`. This is for alpha channel:

In [81]:
print(len([14, 15, 16, 17, 18, 19]),len([20, 21, 22, 23, 24, 25, 26, 27, 28]))

6 9


For beta channel:

In [82]:
print(len([54, 55, 56, 57, 58, 59]),len([60, 61, 62, 63, 64, 65, 66, 67, 68]))

6 9


You can see the relation between the indices via the `data_dim/2=80/2=40` i.e. `54 (beta) = 14 (alpha) + 40` and `68 (beta) = 28 (alpha) + 40`. 

Now that we have found the indices which we can use to extract the KS data, we are in a position to start building the Slater determinants using these information.

There is an extra parameter that was added in the function where the user can specify how many occupied orbitals to consider. This keyword, even though useful, but adds to the level of complexity. So, we select it as `0` as it is required for the function `step2_many_body.reindex_cp2k_sd_states` (and its default value is also `0` in that function) and it is not used anymore (even though present in the code). A better approach was brought in the [active space selection tutorial](../17_active_space_selection).

If it is spin-restricted KS calculations we should have `sd_format = 2` otherwise it is should be `sd_format = 1`.

In [83]:
help(step2_many_body.reindex_cp2k_sd_states)

Help on function reindex_cp2k_sd_states in module libra_py.workflows.nbra.step2_many_body:

reindex_cp2k_sd_states(ks_orbital_homo_index, ks_orbital_indicies, sd_basis_states, sd_format=2, ks_beta_homo_index=0, active_space_num_occ_orbitals=0)
    ks_orbital_homo_index: Index of the homo ks orbital, from 1
    ks_orbital_indicies: Range of the considered ks orbtials. Ex) [8,9,10,11], where 9 is homo orbtial index (from 1)
    sd_basis_states( list of lists of lists ): A list of Slater determinants, where each slater determinant is a excitation in the Kohn-Sham
                                               basis. This function assumes that all Kohn-Sham excitations are for alpha electrons. To
                                               differentiate between alpha and beta excitations, elements of sd_basis_states contain spin
                                               information.
    
                                               Ex) sd_basis_states[0] = [ [9,10], "alp" ]
     

Taking a closer look into this function for the `sd_format` we see that it was mainly designed to generate a correct ground state (because from this GS we build other excited states basis):

```python
# Form ground-state SD first
sd_basis = [[]]
if sd_format == 1:
    for i in range(1, len(ks_orbital_indicies)):
        if i < alp_homo_matrix_index + 2:
            sd_basis[0].append(i)
        if i < beta_homo_matrix_index + 2:
            sd_basis[0].append(-i)
            
elif sd_format == 2:
    for i in range(1, len(ks_orbital_indicies)):
        if i < alp_homo_matrix_index + 2:
            sd_basis[0].append(i)
            sd_basis[0].append(-i)
            
print("ground state = ", sd_basis)

```

and the continuation of this function is as:

```python
for j in range(len(excitations)):
    sd_excitation = []
    for sd_state in sd_basis[0]:
        if sd_state == excitations[j][0]:
            sd_excitation.append(excitations[j][1])
        else:
            sd_excitation.append(sd_state)
    sd_basis.append(sd_excitation)
```

where `excitations` is defined above that as:

```python
if sd_format == 1:
    
    if sd_basis_states[j][1] == "alp":
        initial_ks_orb = int(sd_basis_states[j][0][0]) - ks_orbital_homo_index + alp_homo_matrix_index + 1
        final_ks_orb = int(sd_basis_states[j][0][1]) - ks_orbital_homo_index + alp_homo_matrix_index + 1
    elif sd_basis_states[j][1] == "bet":
        initial_ks_orb = int(sd_basis_states[j][0][0]) - ks_beta_homo_index + beta_homo_matrix_index + 1
        final_ks_orb = int(sd_basis_states[j][0][1]) - ks_beta_homo_index + beta_homo_matrix_index + 1

elif sd_format == 2:
    
    initial_ks_orb = int(sd_basis_states[j][0][0]) - ks_orbital_homo_index + alp_homo_matrix_index + 1
    final_ks_orb = int(sd_basis_states[j][0][1]) - ks_orbital_homo_index + alp_homo_matrix_index + 1

if sd_basis_states[j][1] == "alp":
    
    excitations.append([initial_ks_orb, final_ks_orb])

elif sd_basis_states[j][1] == "bet":
    
    excitations.append([-initial_ks_orb, -final_ks_orb])
```
So, the translation of the above `for` loop is that it find the ground state, then for each element in that, if it finds the occupied orbitals (or the initial state here `excitations[j][0]`) in that list, it will replace it with the target `excitations[j][1]` and append it to the `sd_basis`. 

It looks a bit complicated but it's all index manipulations so that we can get to the correct values. Let's try it:

In [84]:
sd_states_reindexed = step2_many_body.reindex_cp2k_sd_states(ks_homo_index,ks_orbital_indicies,
                                                             sd_unique_basis,sd_format=2,
                                                             active_space_num_occ_orbitals=0)
print('sd_states_reindexed', sd_states_reindexed)
print('First excited state SD (HOMO (6 alpha) -> LUMO (7 alpha)):', sd_states_reindexed[1])

ground state =  [[1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6]]
sd_states_reindexed [[1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 7, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 10, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 8, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 9, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 11, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 12, -6], [1, -1, 2, -2, 3, -3, 4, -4, 7, -5, 6, -6], [1, -1, 2, -2, 3, -3, 4, -4, 8, -5, 6, -6], [1, -1, 2, -2, 7, -3, 4, -4, 5, -5, 6, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 13, -6], [1, -1, 2, -2, 3, -3, 7, -4, 5, -5, 6, -6], [7, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6], [1, -1, 2, -2, 3, -3, 8, -4, 5, -5, 6, -6], [1, -1, 2, -2, 3, -3, 4, -4, 9, -5, 6, -6], [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 15, -6], [1, -1, 2, -2, 8, -3, 4, -4, 5, -5, 6, -6], [1, -1, 2, -2, 3, -3, 9, -4, 5, -5, 6, -6], [1, -1, 8, -2, 3, -3, 4, -4, 5, -5, 6, -6]]
First excited state SD (HOMO (6 alpha) -> LUMO (7 alpha)): [1, -1, 2, -2, 3, 

Okay! We have our set of Slater-determinants representations. To build each of these matrices we will follow the approach brought in the `mapping` or `mapping3` modules. The next step we hould take is to turn the elements of the `sd_states_reindexed` variable into indices so that we can select them from the KS matrix. This is done via the function `mapping.sd2indx`.

In [85]:
help(mapping.sd2indx)

Help on function sd2indx in module libra_py.workflows.nbra.mapping:

sd2indx(inp, nbasis=0, do_sort=False, user_notation=0)
    This function maps a list of integers defining the
    occupied spin-orbitals in a SD onto the global indices of the orbitals
    that are being occupied.
    
    Args:
        inp ( list of ints ): indices of the occupied spin-orbitals.
            Indexing starts with 1, not 0!
    
            Positive number ```n``` correspond to an alpha electron occupying the
            orbital (whether it is an alpha-spatial or beta-spatial component) ```n```
    
            Negative number ```-n``` correspond to a beta electron occupying the
            orbital (whether it is an alpha-spatial or beta-spatial component) ```n```
    
        nbasis ( int ): the total number of orbitals orbitals (both alpha- and beta-
            spatial components ) in the selected active space [currently not really used!]
    
        do_sort ( Boolean ): the flag to tell whether the

First we read a time-overlap matrix in `sample_matrix`. Then, using the `ks_active_space` generated above, we obtain a matrix called `St_ks`. You can see the dimension of this matrix is now `15*2=30`.

In [86]:
sample_matrix = sp.load_npz('St_ks_1200.npz')
St_ks = sample_matrix[ks_active_space,:][:,ks_active_space]
print('St.shape:', St_ks.shape)
SD1 = sd_states_reindexed[1]
print('SD1:', SD1)
sd1 = mapping.sd2indx(SD1, St_ks.shape[0], False, 0) # No use_minimal and no user_notation
# What about beta spin channels? We just need to shift the ones
# related to negative values in SD1 by data_dim/2
beta_indices = np.where(np.array(SD1) < 0)
sd1 = np.array(sd1)
sd1[beta_indices] += int(St_ks.shape[0]/2)
print('sd1 (indices of the KS MO matrix):', sd1)

St.shape: (30, 30)
SD1: [1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 7, -6]
sd1 (indices of the KS MO matrix): [ 0 15  1 16  2 17  3 18  4 19  6 20]


In [87]:
# For example the St_ks[HOMO_alpha, HOMO-1_alpha]
# and St[HOMO_beta, HOMO-1_beta] should be the same as we don't consider the 
# unrestricted spin calculations...
St_ks[5,4], St_ks[20,19]

((-0.00039952943625914256+0j), (-0.00039952943625914256+0j))

You can see that since we have `6` occupied orbitals in our active space, the occupied orbitals starts from `0` to `5`. And for beta channel, they start from `15` to `20`. 

Let's build and compute the time-overlap between Slater-determinant for SD1 for example.

This is exactly as appears in `mapping3.ovlp_arb` function.

Let's build the full matrix from `sd1` as:

In [88]:
Slater_determinant_matrix = St_ks[sd1,:][:,sd1]
print('Slater_determinant_matrix.shape', Slater_determinant_matrix.shape)
print('----------------------')
print('Full Slater determinant:',Slater_determinant_matrix.todense())
print('----------------------')
print('Diagonal elements of the Slater_determinant_matrix:', np.diag(Slater_determinant_matrix.todense()))

Slater_determinant_matrix.shape (12, 12)
----------------------
Full Slater determinant: [[ 9.98647621e-01+0.j  0.00000000e+00+0.j -3.03739417e-02+0.j
   0.00000000e+00+0.j  2.83817695e-02+0.j  0.00000000e+00+0.j
  -2.16160467e-02+0.j  0.00000000e+00+0.j -2.06989970e-02+0.j
   0.00000000e+00+0.j  1.09562451e-03+0.j  0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j  9.98647621e-01+0.j  0.00000000e+00+0.j
  -3.03739417e-02+0.j  0.00000000e+00+0.j  2.83817695e-02+0.j
   0.00000000e+00+0.j -2.16160467e-02+0.j  0.00000000e+00+0.j
  -2.06989970e-02+0.j  0.00000000e+00+0.j -3.05721239e-03+0.j]
 [ 3.18093754e-02+0.j  0.00000000e+00+0.j  9.98719552e-01+0.j
   0.00000000e+00+0.j -3.43856659e-02+0.j  0.00000000e+00+0.j
   1.42843738e-02+0.j  0.00000000e+00+0.j  6.45308637e-03+0.j
   0.00000000e+00+0.j -3.58407930e-04+0.j  0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j  3.18093754e-02+0.j  0.00000000e+00+0.j
   9.98719552e-01+0.j  0.00000000e+00+0.j -3.43856659e-02+0.j
   0.00000000e+00+0.j  1.42843738e-02+0.

Now, just compute the determinant...

In [89]:
res = np.linalg.det(Slater_determinant_matrix.todense())
print(res)

(0.9989976226281522+0j)


Looks reasonable! Let's do the same appoach for the set of all SDs and see what they will return:

In [90]:
nSDs = len(sd_states_reindexed)
SD_time_overlap = np.zeros((nSDs, nSDs))
for i in range(nSDs):
    for j in range(nSDs):
        SD1 = sd_states_reindexed[i]
        sd1 = mapping.sd2indx(SD1, St_ks.shape[0], False, 0)
        SD2 = sd_states_reindexed[j]
        sd2 = mapping.sd2indx(SD2, St_ks.shape[0], False, 0)
        beta_indices = np.where(np.array(SD1) < 0)
        sd1 = np.array(sd1)
        sd1[beta_indices] += int(St_ks.shape[0]/2)
        beta_indices = np.where(np.array(SD2) < 0)
        sd2 = np.array(sd2)
        sd2[beta_indices] += int(St_ks.shape[0]/2)
        Slater_determinant_matrix = St_ks[sd1,:][:,sd2]
        res = np.linalg.det(Slater_determinant_matrix.todense())
        SD_time_overlap[i,j] = res.real

In [93]:
print('SD_time_overlap.shape:', SD_time_overlap.shape)
print('Number of single-particle excitations + 1 (ground state):', len(sd_unique_basis)+1)
print('\nnp.diag(SD_time_overlap):', np.diag(SD_time_overlap))
print('--------------------------------')
# print('And the full SD_time_overlap_matrix:\n', SD_time_overlap)

SD_time_overlap.shape: (19, 19)
Number of single-particle excitations + 1 (ground state): 19

np.diag(SD_time_overlap): [ 0.9991323   0.99899762  0.99891845  0.99897877  0.99884844  0.99905603
  0.99901041  0.99838514  0.99820925 -0.99770115  0.99902244  0.9986569
  0.99774958  0.99848916  0.99807277  0.99898329 -0.99754027  0.99834682
  0.99766752]
--------------------------------


Let's see for example the time-overlaps of the excited states with the ground state i.e. both $|\Psi_{GS}(t)|\Psi_{S_i}(t+\Delta t)\rangle$ and $|\Psi_{S_i}(t)|\Psi_{GS}(t+\Delta t)\rangle$

In [94]:
SD_time_overlap[0,:], SD_time_overlap[:,0]

(array([ 9.99132301e-01,  1.53157211e-02, -1.18267619e-03,  9.60011838e-03,
        -8.81029115e-03,  1.03501713e-04, -1.99158530e-04, -8.77520698e-04,
        -3.35116544e-03, -2.27368800e-03,  1.42099405e-04,  1.43692545e-03,
         1.20272882e-03, -2.07325454e-03,  1.21490579e-03,  5.13753953e-04,
        -9.50620783e-04, -4.69983339e-04,  1.65203837e-04]),
 array([ 9.99132301e-01, -1.55381298e-02,  9.71899842e-04, -9.51487759e-03,
         8.54027099e-03, -1.49363725e-04,  2.32223199e-04,  8.34935923e-04,
         3.32621904e-03, -2.41182930e-03, -1.00635719e-04, -1.41846701e-03,
        -1.11322957e-03,  2.05329921e-03, -1.08416389e-03, -6.02548469e-04,
        -1.05893233e-03,  4.98538862e-04, -2.09867048e-04]))

### Computational time for large number of SDs

Here, we intend to compute how long does it take to compute `SD_time_overlap` matrix if we have a large number of excitations. Well, we can increase this number by reducing the `tolerance` value to something like `0.0`. Let's try from the beginning. I will just copy and paste the above code below with the only difference is that I have changed the `tolerance` value to `0.0`.

**Note:** The output of this cell is cleared for the toturial.

In [96]:
params_2 = {"logfile_directory": "all_logfiles",
            "es_software": "cp2k", "isUKS": 0, "number_of_states": 10,
            "tolerance": 0.0, "isnap": 1200, "fsnap": 1210, 
            'lowest_orbital': lowest_orbital, 'highest_orbital': highest_orbital}

res = step3_many_body.get_step2_mb_sp_properties(params_2)
# The unique SDs in the TD-DFT results in all logfiles
sd_unique_basis = res[0]

# The target KS states in TD-DFT
sd_fstates = []

# The initial KS states in TD-DFT
sd_tstates = []
for i in range(len(sd_unique_basis)):
    sd_fstates.append(sd_unique_basis[i][0][0])
    sd_tstates.append(sd_unique_basis[i][0][1])

# The min_band present in the excitation
min_band = min(sd_fstates)

# The max_band present in the excitation
max_band = max(sd_tstates)

# number of occupied and unoccupied KS states
num_occ = ks_homo_index - min_band + 1
num_unocc = max_band - ks_homo_index  # + 1

print('min_band:', min_band)
print('max_band:', max_band)
print('num_occ:', num_occ)
print('num_unocc:', num_unocc)
ks_orbital_indicies = range(min_band, max_band + 1)
print('The range of the KS orbital indices (starting from 1):', list(ks_orbital_indicies))


# What is the KS HOMO index in the raw npz files
npz_file_ks_homo_index = ks_homo_index - lowest_orbital + 1
# simply read a sample matrix and then see what is its dimension
sample_matrix = sp.load_npz('sample_step2_data/res/St_ks_1200.npz')
data_dim = sample_matrix.shape[0]
print('data_dim:', data_dim)
ks_active_space, ks_homo_index_1 = step3.make_active_space(num_occ, num_unocc, data_dim, npz_file_ks_homo_index)
print('npz_file_ks_homo_index:', npz_file_ks_homo_index)
print('ks_active_space:', ks_active_space)
print('ks_homo_index_1:', ks_homo_index_1)

sd_states_reindexed = step2_many_body.reindex_cp2k_sd_states(ks_homo_index,ks_orbital_indicies,
                                                             sd_unique_basis,sd_format=2,
                                                             active_space_num_occ_orbitals=0)
print('sd_states_reindexed', sd_states_reindexed)
print('First excited state SD (HOMO (20 alpha) -> LUMO (21 alpha)):', sd_states_reindexed[1])
print('Total number of excited states + ground state:', len(sd_states_reindexed))

min_band: 21
max_band: 60
num_occ: 20
num_unocc: 20
The range of the KS orbital indices (starting from 1): [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
data_dim: 80
npz_file_ks_homo_index: 20
ks_active_space: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
ks_homo_index_1: 20
ground state =  [[1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -18, 19, -19, 20, -20]]
sd_states_reindexed [[1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7, 8, -8, 9, -9, 10, -10, 11, -11, 12, -12, 13, -13, 14, -14, 15, -15, 16, -16, 17, -17, 18, -1

We are now going to check how long does it take to compute the time-overlap matrix for 401 number of SDs for a single step.

In [97]:
t1 = time.time()
sample_matrix = sp.load_npz('St_ks_1200.npz')
St_ks = sample_matrix[ks_active_space,:][:,ks_active_space].todense()
print(St_ks.shape)
nSDs = len(sd_states_reindexed)
SD_time_overlap = np.zeros((nSDs, nSDs))
for i in range(nSDs):
    for j in range(nSDs):
        #if i<=j:
        #print('<SD1|SD2>:')
        SD1 = sd_states_reindexed[i]
        sd1 = mapping.sd2indx(SD1, St_ks.shape[0], False, 0)
        SD2 = sd_states_reindexed[j]
        sd2 = mapping.sd2indx(SD2, St_ks.shape[0], False, 0)
        beta_indices = np.where(np.array(SD1) < 0)
        sd1 = np.array(sd1)
        sd1[beta_indices] += int(St_ks.shape[0]/2)
        beta_indices = np.where(np.array(SD2) < 0)
        sd2 = np.array(sd2)
        sd2[beta_indices] += int(St_ks.shape[0]/2)
        #print(sd1)
        #print(sd2)
        Slater_determinant_matrix = St_ks[sd1,:][:,sd2]
        res = np.linalg.det(Slater_determinant_matrix)
        SD_time_overlap[i,j] = res.real
print('Elapsed time:', time.time()-t1)

(80, 80)
Elapsed time: 13.631155490875244


It takes about 14 seconds to compute the time-overlap matrix for 401 SDs for a single step. This is a good timing and we are also doing step 3 calculations for each step using `multiprocessing` library of Python. What about larger number of SDs? Well, for this, we require larger systems with a larger KS MO matrices. Before closing this tutorial, let's check the diagnoal elements of the `SD_time_overlap` to check if they make sense or not?!

In [98]:
print('SD_time_overlap.shape:', SD_time_overlap.shape)
print('Number of single-particle excitations + 1 (ground state):', len(sd_unique_basis)+1)
print('\nnp.diag(SD_time_overlap):', np.diag(SD_time_overlap))

SD_time_overlap.shape: (401, 401)
Number of single-particle excitations + 1 (ground state): 401

np.diag(SD_time_overlap): [ 0.99359512  0.99346023  0.99337354  0.99343994  0.99330667  0.99262064
  0.99289678  0.99351405  0.99290894 -0.99190379  0.99184428  0.99199891
  0.99318695  0.99263886  0.99305284  0.99246651 -0.99194263  0.99272962
  0.99285928  0.99266899  0.99347912  0.99274416  0.99296103  0.99203482
  0.99283926  0.99191781 -0.98867335 -0.99035431  0.98955661 -0.98822514
  0.9929456   0.99252341  0.99335418  0.99259609  0.99303206  0.98354556
 -0.98035419  0.99413687  0.99222842  0.9926108   0.98449478  0.9904014
  0.99203402 -0.99197578  0.99142544  0.99277802  0.99274166  0.98618291
 -0.99209749  0.98974385  0.98221823  0.99288557  0.98614494  0.99214667
  0.99258332  0.99196118 -0.97935742  0.99024636 -0.99183851  0.9823976
  0.99125972  0.98929375  0.98417006  0.99207309  0.99206103  0.99363416
 -0.990172    0.99198805  0.99221611  0.99398067 -0.98409558  0.99246904
  0

Let's see what is the determinant of this matrix:

In [99]:
print('Determinant of the overlap matrix of the Slater determinants:', np.linalg.det(SD_time_overlap))

Determinant of the overlap matrix of the Slater determinants: 0.05403119970851967


This is small but still far from `0.0` - which prohibit the inversion of the matrix in local diabatization approach. 


**Note:** We know that the time-overlaps are close to `1.0` but not equal to `1.0`. So, as the number of single-particle states increases, the determinant of the overlap matrix of the Slater determinants goes to `0.0`. Let's say, we have a diagonal matrix with entries all equal to `1.0-0.01 = 0.99`. As the size of the matrix increases, the determinant goes to zero and hence the inversion is not possible:

In [100]:
a = 0.99
print('Determinant of a diagonal matrix of size 10:', a**10)
print('Determinant of a diagonal matrix of size 400:', a**400)
print('Determinant of a diagonal matrix of size 1000:', a**1000)
print('Determinant of a diagonal matrix of size 10000:', a**10000)

Determinant of a diagonal matrix of size 10: 0.9043820750088044
Determinant of a diagonal matrix of size 400: 0.017950553275045137
Determinant of a diagonal matrix of size 1000: 4.317124741065786e-05
Determinant of a diagonal matrix of size 10000: 2.2487748498162805e-44
